In [11]:
!pip install pennylane

In [12]:
import pennylane as qml
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

n_qubits = 4
dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev, interface="torch")
def quantum_circuit(inputs, weights):
    for i in range(n_qubits):
        qml.RY(inputs[i], wires=i)

    qml.BasicEntanglerLayers(weights, wires=range(n_qubits))
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

In [13]:
class QuanvolutionLayer(nn.Module):
    def __init__(self):
        super(QuanvolutionLayer, self).__init__()
        self.q_weights = nn.Parameter(0.01 * torch.rand((1, n_qubits)))  # Quantum trainable weights

    def forward(self, x):
        # apply quanvolution to each 2x2 patch in the input grid
        batch_size, height, width, channels = x.shape
        output = torch.zeros(batch_size, height - 1, width - 1, n_qubits)

        for i in range(height - 1):
            for j in range(width - 1):
                for b in range(batch_size):
                    patch = x[b, i:i+2, j:j+2, :].reshape(-1)  # Extract 2x2 patch
                    q_out = torch.tensor(quantum_circuit(patch, self.q_weights), dtype=torch.float32).detach()  #features
                    output[b, i, j, :] = torch.tensor(q_out)

        return output

In [14]:
class QuanvolutionalNN(nn.Module):
    def __init__(self):
        super(QuanvolutionalNN, self).__init__()
        self.quanv = QuanvolutionLayer()
        self.conv = nn.Conv2d(in_channels=n_qubits, out_channels=16, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(16 * 6 * 6, 64)
        self.fc2 = nn.Linear(64, 1)

    def forward(self, x):
        x = self.quanv(x)
        x = x.permute(0, 3, 1, 2)
        x = torch.relu(self.conv(x))
        x = torch.flatten(x, start_dim=1)
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [15]:
num_samples = 100
data = np.load('input_aug.npy')
labels = np.load('outputpH.npy')
print(data.shape, labels.shape)
labels = torch.tensor(labels, dtype=torch.float32)

(1062, 7, 7, 6) (1062, 6, 1)


I took the first depth only (one model to each depth)

In [16]:
labels = labels[:, 0, :]
print(labels.shape)
print(labels)

torch.Size([1062, 1])
tensor([[7.8700],
        [7.8700],
        [7.8700],
        ...,
        [6.3310],
        [6.3310],
        [6.3310]])


In [17]:
model = QuanvolutionalNN()
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

num_epochs = 20
for epoch in range(num_epochs):
    optimizer.zero_grad()
    outputs = model(data)
    print(type(outputs), type(labels))
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()
    print(f"epoch {epoch+1}/{num_epochs}\n loss: {loss.item():.4f}")

<ipython-input-13-d0b1d429b070>:16: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  output[b, i, j, :] = torch.tensor(q_out)


<class 'torch.Tensor'> <class 'torch.Tensor'>
epoch 1/20
 loss: 61.1131
<class 'torch.Tensor'> <class 'torch.Tensor'>
epoch 2/20
 loss: 47.5081
<class 'torch.Tensor'> <class 'torch.Tensor'>
epoch 3/20
 loss: 31.5031
<class 'torch.Tensor'> <class 'torch.Tensor'>
epoch 4/20
 loss: 14.9825
<class 'torch.Tensor'> <class 'torch.Tensor'>
epoch 5/20
 loss: 6.0720
<class 'torch.Tensor'> <class 'torch.Tensor'>
epoch 6/20
 loss: 14.9745
<class 'torch.Tensor'> <class 'torch.Tensor'>
epoch 7/20
 loss: 18.8454
<class 'torch.Tensor'> <class 'torch.Tensor'>
epoch 8/20
 loss: 12.5792
<class 'torch.Tensor'> <class 'torch.Tensor'>
epoch 9/20
 loss: 6.4750
<class 'torch.Tensor'> <class 'torch.Tensor'>
epoch 10/20
 loss: 4.6217
<class 'torch.Tensor'> <class 'torch.Tensor'>
epoch 11/20
 loss: 5.7780
<class 'torch.Tensor'> <class 'torch.Tensor'>
epoch 12/20
 loss: 7.5909
<class 'torch.Tensor'> <class 'torch.Tensor'>
epoch 13/20
 loss: 8.6639
<class 'torch.Tensor'> <class 'torch.Tensor'>
epoch 14/20
 loss: 8